# Transform the Olist data

Builds the clean Delta tables the semantic model uses. Run the cells **top to bottom** in a Fabric notebook attached to the `olist_lakehouse` Lakehouse.

It expects the raw tables loaded from the Kaggle CSVs: `orders`, `products`, `productcategorynametranslation`, `geolocation` and `orderreviews`.

| Output table | What it holds |
|---|---|
| `orders_clean` | orders with real timestamps, `delivery_duration_days`, `on_time` and `order_purchase_date` |
| `products_clean` | products joined to their English category names |
| `geolocation_clean` | one latitude/longitude (the average) per zip code prefix |
| `reviews_clean` | reviews with an integer `review_score` (rows without a valid score removed) |
| `dim_date` | one row per day, 2016-01-01 to 2018-12-31 |

## 1. Orders

In [ ]:
from pyspark.sql.functions import col, to_timestamp, to_date, datediff, when

orders = spark.table("orders")

# The date columns are loaded as text, so convert them to timestamps
date_cols = [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date"
]
for c in date_cols:
    orders = orders.withColumn(c, to_timestamp(col(c)))

orders = (
    orders
    # whole days between purchase and delivery (null when the order was never delivered)
    .withColumn(
        "delivery_duration_days",
        datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp"))
    )
    # 1 when delivered on or before the estimated date, otherwise 0.
    # NOTE: an order that was never delivered has no delivery date, the comparison is not true, and it also gets 0.
    # See "Known limitations" in the README.
    .withColumn(
        "on_time",
        when(col("order_delivered_customer_date") <= col("order_estimated_delivery_date"), 1).otherwise(0)
    )
    # the calendar date of the purchase, used to relate each order to dim_date
    .withColumn("order_purchase_date", to_date("order_purchase_timestamp"))
)

# overwriteSchema lets a re-run replace the table even though its columns changed
orders.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("orders_clean")

## 2. Products

In [ ]:
products = spark.table("products")
translation = spark.table("productcategorynametranslation")

# left join keeps products that have no translated category name
products_clean = products.join(translation, on="product_category_name", how="left")
products_clean.write.mode("overwrite").format("delta").saveAsTable("products_clean")

## 3. Geolocation

The raw table has many rows per zip code prefix; keep one average point for each.

In [ ]:
from pyspark.sql.functions import avg

geo = spark.table("geolocation")
geo_clean = geo.groupBy("geolocation_zip_code_prefix").agg(
    avg("geolocation_lat").alias("lat"),
    avg("geolocation_lng").alias("lng")
)
geo_clean.write.mode("overwrite").format("delta").saveAsTable("geolocation_clean")

## 4. Reviews

`review_score` is loaded as text. A blank string is **not** null, so a plain null check misses it. Casting to an integer turns blanks (and any other non-numeric text) into null, and those rows are then dropped. In the run documented in the README this removed 2,555 blank scores on top of 2,380 rows that were already null, leaving 99,227 reviews.

In [ ]:
reviews = spark.table("orderreviews")

reviews_clean = (
    reviews
    .withColumn("review_score", col("review_score").cast("int"))
    .filter(col("review_score").isNotNull())
)
reviews_clean.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("reviews_clean")

print(f"dropped {reviews.count() - reviews_clean.count()} rows with a missing or non-numeric review_score")

## 5. Date table

A calendar table is needed for the time-intelligence measures (`SAMEPERIODLASTYEAR`, `DATEADD`). The range covers the order dates in this dataset; widen it if you load newer data.

In [ ]:
from pyspark.sql.functions import explode, year, month, quarter, date_format

date_range = spark.sql("SELECT sequence(to_date('2016-01-01'), to_date('2018-12-31'), interval 1 day) as date")
dim_date = (
    date_range.select(explode(col("date")).alias("Date"))
    .withColumn("Year", year("Date"))
    .withColumn("Month", month("Date"))
    .withColumn("MonthName", date_format("Date", "MMMM"))
    .withColumn("Quarter", quarter("Date"))
)

dim_date.write.mode("overwrite").format("delta").saveAsTable("dim_date")

## 6. Check the results

Row counts and the final column types of the two tables whose schema changed.

In [ ]:
for t in ["orders_clean", "products_clean", "geolocation_clean", "reviews_clean", "dim_date"]:
    print(t, spark.table(t).count())

spark.table("orders_clean").printSchema()
spark.table("reviews_clean").printSchema()

Row counts from the run documented in the README: `orders_clean` 99,441, `products_clean` 32,951, `geolocation_clean` 19,015, `reviews_clean` 99,227, `dim_date` 1,096.